# 02 — Normalise the extract into keyed tables and load them into SQLite

**Reads:** `data/processed/psid_wide.parquet`, `data/raw/psid_variable_crosswalk.csv`, `data/raw/cpi_u_annual.csv`.
**Writes:** `data/processed/psid.sqlite` with three tables, plus `results/tables/table_counts.csv`.

The Data Center pastes each year's family record onto every member's row, so one wide row per person holds
43 waves of both person-level and family-level columns. This notebook undoes that into the tables PSID
conceptually has:

| table | one row per | columns |
|---|---|---|
| `individuals` | person | `id_1968, person_num, sex, father_id_1968, father_person_num, mother_id_1968, mother_person_num` |
| `persons` | person × year | `id_1968, person_num, year, interview, age, rel_head, seq` |
| `families` | family × year | `year, fam_interview, fam_income, income_year, fam_income_real` |

Each step ends with a count — the Class 12 rule: validate row counts and key uniqueness after every reshape or join.

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

ROOT = Path.cwd().parent
RAW, PROC, TAB = ROOT / "data" / "raw", ROOT / "data" / "processed", ROOT / "results" / "tables"
TAB.mkdir(parents=True, exist_ok=True)

wide = pd.read_parquet(PROC / "psid_wide.parquet")
cw = pd.read_csv(RAW / "psid_variable_crosswalk.csv")
cpi = pd.read_csv(RAW / "cpi_u_annual.csv")
wide.shape

(85536, 264)

## 1. `individuals` — the fixed columns
Rows = people. Key = (`id_1968`, `person_num`). Check the key is unique before anything else.

In [2]:
individuals = wide[[
    "id_1968",
    "person_num",
    "sex",
    "father_id_1968",
    "father_person_num",
    "mother_id_1968",
    "mother_person_num"
]]

duplicate_keys = individuals.duplicated(subset=["id_1968", "person_num"]).sum()
assert duplicate_keys == 0

individuals.shape
individuals

,id_1968,person_num,sex,father_id_1968,father_person_num,mother_id_1968,mother_person_num
0,1,1,1,0,0,0,0
1,1,2,2,0,0,0,0
2,1,3,1,0,0,0,0
3,1,4,2,0,0,0,0
4,1,30,2,0,0,0,0
...,...,...,...,...,...,...,...
85531,9308,1,1,0,0,0,0
85532,9308,2,2,0,0,0,0
85533,9308,3,1,0,0,9308,2
85534,9308,4,2,0,0,9308,2


## 2. `persons` — wide → long on the four per-year person variables
Target: one row per person × year with `interview, age, rel_head, seq`. The crosswalk gives you, for each
column name, its `concept` and `year`; stack the 43 wave-blocks. Keep only rows where the person was in the family unit at
that year's interview: `seq` 1–20 (`interview > 0` is not enough, see the comment in the code). 1968 has no `seq`; there, `rel_head > 0`. Then check: how many rows, and is (`id_1968`, `person_num`, `year`) unique?

In [3]:
cw.groupby(["table", "concept"]).size()

table       concept                     
family      fam_interview_number            43
            fam_total_family_income         43
individual  father_1968_interview_number     1
            father_person_number             1
            ind_age                         43
            ind_id_1968_interview_number     1
            ind_id_person_number             1
            ind_interview_number            42
            ind_relationship_to_head        43
            ind_sequence_number             42
            ind_sex                          1
            mother_1968_interview_number     1
            mother_person_number             1
dtype: int64

In [4]:
years = sorted(cw.loc[cw["concept"] == "ind_age", "year"].astype(int))

blocks = []
for y in years:
    block = wide[["id_1968", "person_num", f"interview_{y}", f"age_{y}", f"rel_head_{y}"]].copy()
    block = block.rename(columns={
        f"interview_{y}": "interview",
        f"age_{y}": "age",
        f"rel_head_{y}": "rel_head",
    })
    block["seq"] = wide[f"seq_{y}"] if f"seq_{y}" in wide else pd.NA   # 1968 has no sequence number
    block["year"] = y
    blocks.append(block)

persons_all = pd.concat(blocks, ignore_index=True)
persons_all["seq"] = persons_all["seq"].astype("Int64")
print("person-years before filtering:", len(persons_all))

# Present = living in the family unit at that year's interview: seq 1-20 (codebook).
# interview > 0 is not enough: the extract carries non-zero interview numbers for people with seq = 0
# (e.g. not yet born). 1968 has no seq, so presence there is rel_head > 0.
in_fu = persons_all["seq"].between(1, 20).fillna(False)
in_fu_1968 = (persons_all["year"] == 1968) & (persons_all["rel_head"] > 0)
print("interview > 0 but not in the family unit:", ((persons_all["interview"] > 0) & ~(in_fu | in_fu_1968)).sum())

persons = (persons_all[in_fu | in_fu_1968]
           [["id_1968", "person_num", "year", "interview", "age", "rel_head", "seq"]]
           .reset_index(drop=True))

duplicate_keys = persons.duplicated(subset=["id_1968", "person_num", "year"]).sum()
assert duplicate_keys == 0
assert (persons["interview"] > 0).all()
print("persons:", persons.shape)
persons.groupby("year").size().iloc[[0, 1, 22, -1]]

person-years before filtering: 3678048
interview > 0 but not in the family unit: 1147991
persons: (910869, 7)


year
1968    18230
1969    16493
1990    26930
2023    23283
dtype: int64

## 3. `families` — wide → long on the two per-year family variables, then dedupe
Target: one row per family × year. After melting, the same family-year appears once per member; drop the repeats.
Then the uniqueness check on (`year`, `fam_interview`), and a check that every person-year finds its family record.
Add `income_year = year - 1` and `fam_income_real` using the CPI deflator for the income year.

In [5]:
fam_years = sorted(cw.loc[cw["concept"] == "fam_interview_number", "year"].astype(int))

blocks = []
for y in fam_years:
    block = wide[["id_1968", "person_num", f"fam_interview_{y}", f"fam_income_{y}"]].copy()
    block = block.rename(columns={f"fam_interview_{y}": "fam_interview", f"fam_income_{y}": "fam_income"})
    block["year"] = y
    blocks.append(block)
member_years = pd.concat(blocks, ignore_index=True)

# Keep a family-year only through members who are in `persons` that year, then one row per family-year
member_years = member_years.merge(persons[["id_1968", "person_num", "year"]],
                                  on=["id_1968", "person_num", "year"], how="inner")
print("member-years with a family record:", len(member_years))
families = (member_years[["year", "fam_interview", "fam_income"]]
            .drop_duplicates()
            .astype({"fam_interview": "int64"})
            .reset_index(drop=True))

dupes = families[families.duplicated(subset=["year", "fam_interview"], keep=False)]
print("family-years with more than one income:", len(dupes))
assert dupes.empty

# Every person-year finds its family record: person's interview number == family interview number
link = persons.merge(families, left_on=["year", "interview"], right_on=["year", "fam_interview"],
                     how="left", indicator=True)
print(link["_merge"].value_counts().to_dict())
assert (link["_merge"] == "both").all()

# Income reported in survey year t is for calendar year t - 1: deflate with the income year's CPI
families["income_year"] = families["year"] - 1
families = families.merge(cpi[["year", "deflator_to_base"]].rename(columns={"year": "income_year"}),
                          on="income_year", how="left")
assert families["deflator_to_base"].notna().all()
families["fam_income_real"] = families["fam_income"] * families["deflator_to_base"]
families = families.drop(columns="deflator_to_base")

print("families:", families.shape, "| missing income:", families["fam_income"].isna().sum(),
      "| negative income:", (families["fam_income"] < 0).sum())
families.head()

member-years with a family record: 910869
family-years with more than one income: 0
{'both': 910869, 'left_only': 0, 'right_only': 0}
families: (320456, 5) | missing income: 0 | negative income: 117


,year,fam_interview,fam_income,income_year,fam_income_real
0,1968,1,13108.0,1967,119671.817948
1,1968,2,2604.0,1967,23773.681258
2,1968,3,1770.0,1967,16159.529888
3,1968,4,6434.0,1967,58740.347625
4,1968,5,7900.0,1967,72124.455431


**Decisions.** `persons` keeps person-years with `seq` 1–20 (in the family at the interview); people in institutions (51–59: college dorms, military, prison), movers-out (71–80) and the deceased (81–89) are dropped for that year. Family records are taken only through members in `persons`, so every family-year has at least one member and the (`year`, `fam_interview`) key holds in all 43 waves. 117 family-years report negative income (business or farm losses) — kept here, handled when logs are taken in notebook 03.

## 4. Load into SQLite and record the counts
`sqlite3.connect(PROC / "psid.sqlite")`, `DataFrame.to_sql(..., if_exists="replace", index=False)`.
Save one small table of row counts per SQL table to `results/tables/table_counts.csv` — it goes in the post.

In [6]:
con = sqlite3.connect(PROC / "psid.sqlite")
tables = {"individuals": individuals, "persons": persons, "families": families}
for name, df in tables.items():
    df.to_sql(name, con, if_exists="replace", index=False)

# Count from the database itself, so the check covers the load too
counts = pd.DataFrame({
    "table": list(tables),
    "rows": [con.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0] for name in tables],
})
assert (counts["rows"].values == [len(df) for df in tables.values()]).all()
con.close()

counts.to_csv(TAB / "table_counts.csv", index=False)
counts

,table,rows
0,individuals,85536
1,persons,910869
2,families,320456
